# Spatial Analytics for Disease Surveillance

## Vertica + DDC Dengue Outbreak Response

---

**Half-Day Workshop** | DDC Training Program

Duration: 3.5 hours (09:00 - 12:30)

## Agenda

| Time | Session | What You'll Do |
|------|---------|----------------|
| 09:00-09:20 | **Introduction** | Why in-database spatial? |
| 09:20-10:15 | **Module 1** | Distance queries + GEOGRAPHY |
| 10:15-10:30 | **Break** | |
| 10:30-11:15 | **Module 2** | Risk buffers for fogging teams |
| 11:15-11:50 | **Module 3** | Heatmaps for situation reports |
| 11:50-12:15 | **Hands-on** | Exercises (you write SQL!) |
| 12:15-12:30 | **Wrap-up** | Visualization + Q&A |

## The Problem

DDC receives dengue case reports with **GPS coordinates**.

Questions that need answers **today, not next week:**

1. Which **schools** are near the outbreak?
2. Which **hospitals** should prepare for severe cases?
3. Where should we send **fogging teams**?
4. How is the outbreak **spreading** over time?

## The Old Way vs. The Vertica Way

| | Old Way | Vertica Way |
|---|---------|-------------|
| **Tool** | Export to QGIS / ArcGIS | SQL in the database |
| **Speed** | Minutes to hours | Milliseconds |
| **Data size** | Crashes on 1M+ rows | Handles billions |
| **Reproducible** | Click-click-click | One query, same result every time |
| **Sharing** | Send shapefiles around | Everyone queries the same DB |

> "คุณไม่ต้อง export ข้อมูลไป GIS อีกต่อไป — GIS มาหาคุณแล้ว"
>
> *You don't need to export data to GIS anymore — GIS comes to you.*

## Our Dataset: 2024 Dengue Season

| Table | Rows | What |
|-------|------|------|
| `dengue_cases` | 30 | Confirmed cases with GPS + severity |
| `hospitals` | 15 | Healthcare facilities |
| `schools` | 10 | At-risk student populations |
| `province_boundaries` | 3 | Bangkok, Chiang Mai, Chiang Rai |

**4 outbreak clusters:**
- Bangkok: Khlong Toei (12 cases), Din Daeng (8 cases)
- Chiang Mai: Old City (6 cases)
- Chiang Rai: Mae Sai border (4 cases)

## Severity Levels (WHO Classification)

| Code | Name | Severity | In Our Data |
|------|------|----------|-------------|
| **DF** | Dengue Fever | Mild | 20 cases (67%) |
| **DHF** | Dengue Hemorrhagic Fever | Severe | 7 cases (23%) |
| **DSS** | Dengue Shock Syndrome | Critical | 3 cases (10%) |

Children (age 0-12) are overrepresented in severe cases.

## Data Model: Spatial Joins, Not Foreign Keys

```
  dengue_cases ──ST_Distance──> hospitals
       │                            │
       │                            │
  ST_DWithin                  ST_Intersects
       │                            │
       v                            v
    schools            province_boundaries
```

Tables are connected through **spatial predicates**, not foreign keys.

All spatial columns use **GEOGRAPHY** type = distances in **meters**.

# Module 1
# Distance Queries
## "Which schools are near the outbreak?"

## The WOW Moment: One Query

```sql
SELECT
    s.name            AS school_name,
    s.district,
    COUNT(*)          AS cases_within_1km,
    ROUND(MIN(ST_Distance(s.geog, d.geog)), 0)
                      AS nearest_case_meters
FROM schools s
JOIN dengue_cases d
    ON ST_Distance(s.geog, d.geog) <= 1000
GROUP BY s.name, s.district
ORDER BY cases_within_1km DESC;
```

One query. Real distances in meters. No GIS export needed.

In [ ]:
from ddc_helpers import run_query, show_on_map, show_buffers, show_heatmap

run_query("""
SELECT
    s.name AS school_name, s.district,
    COUNT(*) AS cases_within_1km,
    SUM(CASE WHEN d.severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_cases,
    ROUND(MIN(ST_Distance(s.geog, d.geog)), 0) AS nearest_case_meters
FROM schools s
JOIN dengue_cases d ON ST_Distance(s.geog, d.geog) <= 1000
GROUP BY s.name, s.district
ORDER BY cases_within_1km DESC
""")

## GEOGRAPHY vs GEOMETRY

| | GEOGRAPHY | GEOMETRY |
|---|-----------|----------|
| **Surface** | Earth (curved) | Flat plane |
| **Distance** | **Meters** | Degrees |
| **Accuracy** | Correct across Thailand | Distorts over large areas |
| **DDC Use** | Nationwide surveillance | Single building layout |

### Example: Bangkok to Chiang Mai

- **GEOGRAPHY:** 583,000 meters = 583 km ✓
- **GEOMETRY:** 5.36 degrees ✗ (meaningless)

> **Rule: DDC surveillance = always GEOGRAPHY**

In [ ]:
# GEOGRAPHY gives METERS (correct)
run_query("""
SELECT
    'GEOGRAPHY' AS type,
    ROUND(ST_Distance(
        ST_GeographyFromText('POINT(100.4856 13.7590)'),
        ST_GeographyFromText('POINT(98.9720 18.7880)')
    ), 0) AS distance,
    'meters' AS unit
UNION ALL
SELECT
    'GEOMETRY',
    ROUND(ST_Distance(
        ST_GeomFromText('POINT(100.4856 13.7590)', 4326),
        ST_GeomFromText('POINT(98.9720 18.7880)', 4326)
    )::NUMERIC, 4),
    'degrees (useless!)'
""")

## Nearest Hospital Per Case

**Why it matters:** If a DSS patient is 30 km from the nearest hospital, DDC needs mobile treatment units.

**Pattern:** CROSS JOIN + ROW_NUMBER()

```sql
SELECT case_id, severity, nearest_hospital, distance_meters
FROM (
    SELECT d.case_id, d.severity, h.name AS nearest_hospital,
        ROUND(ST_Distance(d.geog, h.geog), 0) AS distance_meters,
        ROW_NUMBER() OVER (
            PARTITION BY d.case_id
            ORDER BY ST_Distance(d.geog, h.geog)
        ) AS rn
    FROM dengue_cases d CROSS JOIN hospitals h
) ranked
WHERE rn = 1
ORDER BY distance_meters DESC;
```

In [ ]:
run_query("""
SELECT case_id, patient_age, severity, nearest_hospital, distance_meters
FROM (
    SELECT d.case_id, d.patient_age, d.severity, h.name AS nearest_hospital,
        ROUND(ST_Distance(d.geog, h.geog), 0) AS distance_meters,
        ROW_NUMBER() OVER (PARTITION BY d.case_id ORDER BY ST_Distance(d.geog, h.geog)) AS rn
    FROM dengue_cases d CROSS JOIN hospitals h
) ranked
WHERE rn = 1
ORDER BY distance_meters DESC
""")

In [ ]:
# Visualize: Cases + Hospitals on the map
show_on_map("""
SELECT 'Case ' || case_id AS label, severity,
    ST_Y(geog) AS lat, ST_X(geog) AS lon
FROM dengue_cases
""", lat_col="lat", lon_col="lon", color="red", title="Dengue Cases (red) + Hospitals (blue)")

In [ ]:
show_on_map("""
SELECT name, province,
    ST_Y(geog) AS lat, ST_X(geog) AS lon
FROM hospitals
""", lat_col="lat", lon_col="lon", color="blue")

## Module 1 Key Functions

| Function | Returns | Use |
|----------|---------|-----|
| `ST_GeographyFromText('POINT(lon lat)')` | GEOGRAPHY | Create a point |
| `ST_Distance(a, b)` | meters | Distance between two points |
| `ST_X(geog)` / `ST_Y(geog)` | number | Extract lon / lat |
| `ST_AsText(geog)` | text | Convert to readable WKT |

> **Key pattern:** `JOIN ... ON ST_Distance(a.geog, b.geog) <= N`

# Module 2
# Risk Buffers
## "Where should DDC deploy fogging teams?"

## DDC Risk Tiers for Vector Control

| Tier | Radius | Action | Response Time |
|------|--------|--------|---------------|
| **RED** | 0 - 500m | Immediate fogging + larval survey | Same day |
| **ORANGE** | 500m - 1km | Fogging deployment | Within 48 hours |
| **YELLOW** | 1km - 2km | Enhanced surveillance + community awareness | Within 1 week |

We need to **draw these rings** around every case and figure out:
1. Which schools fall inside?
2. What's the total area to cover?
3. How much do buffers overlap? (saves resources)

## ST_Buffer: Drawing Circles Around Cases

**Important:** `ST_Buffer` works with GEOMETRY (flat), not GEOGRAPHY.

We convert: `ST_GeomFromText(ST_AsText(geog))`

Buffer distance is in **degrees** (at Thai latitude: 1° ≈ 111 km).

```sql
-- 500m buffer = 500/111000 degrees
SELECT
    case_id, severity,
    ST_AsText(
        ST_Buffer(
            ST_GeomFromText(ST_AsText(geog)),
            500.0 / 111000
        )
    ) AS buffer_wkt
FROM dengue_cases
WHERE case_id = 1;
```

## Schools in Danger Zones

Assign a risk tier to each school based on nearest case distance:

```sql
SELECT s.name, d.case_id, d.severity,
    ROUND(ST_Distance(s.geog, d.geog), 0) AS distance_m,
    CASE
        WHEN ST_Distance(s.geog, d.geog) <= 500  THEN 'RED'
        WHEN ST_Distance(s.geog, d.geog) <= 1000 THEN 'ORANGE'
        WHEN ST_Distance(s.geog, d.geog) <= 2000 THEN 'YELLOW'
    END AS risk_tier
FROM schools s
JOIN dengue_cases d
    ON ST_Distance(s.geog, d.geog) <= 2000
ORDER BY distance_m;
```

In [ ]:
# Schools with risk scoring
run_query("""
SELECT
    s.name AS school_name,
    COUNT(*) AS threatening_cases,
    SUM(CASE WHEN ST_Distance(s.geog, d.geog) <= 500  THEN 1 ELSE 0 END) AS red_cases,
    SUM(CASE WHEN ST_Distance(s.geog, d.geog) <= 1000 THEN 1 ELSE 0 END) AS orange_cases,
    SUM(CASE WHEN ST_Distance(s.geog, d.geog) <= 2000 THEN 1 ELSE 0 END) AS yellow_cases,
    ROUND(MIN(ST_Distance(s.geog, d.geog)), 0) AS nearest_case_m
FROM schools s
JOIN dengue_cases d ON ST_Distance(s.geog, d.geog) <= 2000
GROUP BY s.name
ORDER BY threatening_cases DESC
""")

## Merging Overlapping Buffers: ST_Union

Cases in the same cluster have **overlapping** buffers.

`ST_Union()` merges them into a single operational zone.

```sql
SELECT
    pb.province_name,
    ST_AsText(ST_Union(
        ST_Buffer(ST_GeomFromText(ST_AsText(d.geog)),
                  500.0 / 111000)
    )) AS merged_red_zone
FROM dengue_cases d
JOIN province_boundaries pb
    ON ST_Intersects(pb.boundary, d.geog)
GROUP BY pb.province_name;
```

**Result:** One polygon per province instead of 30 overlapping circles.

In [ ]:
# Overlap savings: how much area is saved by merging?
run_query("""
SELECT
    pb.province_name,
    COUNT(d.case_id) AS cases,
    ROUND(SUM(ST_Area(ST_Buffer(ST_GeomFromText(ST_AsText(d.geog)), 500.0/111000))) * 11988, 2)
        AS naive_area_sq_km,
    ROUND(ST_Area(ST_Union(ST_Buffer(ST_GeomFromText(ST_AsText(d.geog)), 500.0/111000))) * 11988, 2)
        AS merged_area_sq_km,
    ROUND((1 - ST_Area(ST_Union(ST_Buffer(ST_GeomFromText(ST_AsText(d.geog)), 500.0/111000)))
        / NULLIFZERO(SUM(ST_Area(ST_Buffer(ST_GeomFromText(ST_AsText(d.geog)), 500.0/111000))))) * 100, 1)
        AS overlap_savings_pct
FROM dengue_cases d
JOIN province_boundaries pb ON ST_Intersects(pb.boundary, d.geog)
GROUP BY pb.province_name
ORDER BY cases DESC
""")

In [ ]:
# Visualize risk buffers on map
show_buffers("""
SELECT
    pb.province_name,
    tier.risk_tier,
    tier.radius_meters,
    COUNT(DISTINCT d.case_id) AS case_count,
    ST_AsText(ST_Union(
        ST_Buffer(ST_GeomFromText(ST_AsText(d.geog)), tier.radius_meters / 111000.0)
    )) AS wkt
FROM dengue_cases d
JOIN province_boundaries pb ON ST_Intersects(pb.boundary, d.geog)
CROSS JOIN (
    SELECT 'RED' AS risk_tier, 500 AS radius_meters
    UNION ALL SELECT 'ORANGE', 1000
    UNION ALL SELECT 'YELLOW', 2000
) tier
GROUP BY pb.province_name, tier.risk_tier, tier.radius_meters
ORDER BY tier.radius_meters DESC
""")

## Module 2 Key Functions

| Function | Returns | Use |
|----------|---------|-----|
| `ST_Buffer(geom, degrees)` | GEOMETRY polygon | Circle around a point |
| `ST_Union(geom)` | GEOMETRY polygon | Merge overlapping shapes |
| `ST_Area(geom)` | sq. degrees | Area of polygon |
| `ST_Intersects(a, b)` | boolean | Do two shapes overlap? |
| `ST_GeomFromText(ST_AsText(geog))` | GEOMETRY | Convert GEOGRAPHY to GEOMETRY |

> **Key pattern:** Buffer → Union → Area → operational deployment zones

# Module 3
# Spatial Heatmaps
## "DDC Weekly Situation Report"

## GeoHash: Spatial Binning

`ST_GeoHash()` converts a point into a **grid cell code**.

Truncating the code controls cell size:

| Precision | Cell Size | Use Case |
|-----------|-----------|----------|
| 5 chars | ~5 km | National overview |
| 6 chars | ~1.2 km | City-level sitrep |
| 7 chars | ~150 m | Neighborhood fogging map |

```sql
SELECT case_id,
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6)
        AS cell_id
FROM dengue_cases;
```

Cases in the same cluster share the same cell code!

In [ ]:
# Cases per grid cell at precision 6
run_query("""
SELECT
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6) AS cell_id,
    COUNT(*) AS case_count,
    SUM(CASE WHEN severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_count,
    MIN(infection_date) AS first_case,
    MAX(infection_date) AS last_case
FROM dengue_cases
GROUP BY cell_id
ORDER BY case_count DESC
""")

## Adding Time: Outbreak Progression

Combine GeoHash binning with `DATE_TRUNC` to track monthly spread:

```sql
SELECT
    DATE_TRUNC('month', infection_date) AS report_month,
    SUBSTR(ST_GeoHash(...), 1, 6)      AS cell_id,
    COUNT(*)                            AS case_count
FROM dengue_cases
GROUP BY report_month, cell_id
ORDER BY report_month, case_count DESC;
```

Watch the outbreak **move** month by month.

In [ ]:
# Monthly progression
run_query("""
SELECT
    DATE_TRUNC('month', infection_date) AS report_month,
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6) AS cell_id,
    COUNT(*) AS case_count,
    SUM(CASE WHEN severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_count
FROM dengue_cases
GROUP BY report_month, cell_id
ORDER BY report_month, case_count DESC
""")

In [ ]:
# Heatmap visualization
show_heatmap("""
SELECT
    SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6) AS cell_id,
    COUNT(*) AS case_count,
    SUM(CASE WHEN severity IN ('DHF','DSS') THEN 1 ELSE 0 END) AS severe_count,
    ST_AsText(ST_GeomFromGeoHash(
        SUBSTR(ST_GeoHash(ST_GeomFromText(ST_AsText(geog))), 1, 6)
    )) AS wkt
FROM dengue_cases
GROUP BY cell_id
ORDER BY case_count DESC
""", title="Dengue Heatmap (Precision 6)")

## Module 3 Key Functions

| Function | Returns | Use |
|----------|---------|-----|
| `ST_GeoHash(geom)` | string | Grid cell code |
| `SUBSTR(hash, 1, N)` | string | Control cell size |
| `ST_GeomFromGeoHash(hash)` | GEOMETRY | Cell boundary for mapping |
| `DATE_TRUNC('month', date)` | date | Time binning |

> **Key pattern:** GeoHash + GROUP BY = instant heatmap

# Hands-On Exercises

## Open `00_workshop_runner.ipynb`

4 exercises, ~30 minutes total:

1. **Proximity Analysis** — Find 3 nearest hospitals to Khlong Toei
2. **Risk Scoring** — Weighted risk score for schools
3. **Heatmap Resolution** — Compare precision 5/6/7
4. **Outbreak Timeline** — Monthly trend analysis

Write your SQL in the empty cells. Run them to see results!

**Hint cells** are available if you get stuck.

# Wrap-Up

## What You Learned Today

| Skill | SQL Pattern |
|-------|-------------|
| Distance queries | `ST_Distance(a.geog, b.geog) <= N` |
| Risk zones | `ST_Buffer()` + `ST_Union()` |
| Heatmaps | `ST_GeoHash()` + `GROUP BY` |
| Time tracking | `DATE_TRUNC()` + spatial binning |

All powered by **GEOGRAPHY** type = automatic meter-based distances.

## From SQL to Decision

```
   SQL Query              Vertica Result         DDC Action
   ─────────              ──────────────         ──────────
   ST_Distance            Schools at risk   -->  Notify schools
   ST_Buffer + Union      Fogging zones     -->  Deploy teams
   ST_GeoHash + time      Monthly heatmap   -->  Situation report
```

Every query you wrote today maps to a **real operational decision**.

## Next Steps

- **Course 2:** Building a Spatial Data Warehouse
  - Star schema, 5M rows, projections for performance
  - SCD Type 2 for boundary changes
- **Connect to BI tools:** Power BI (ODBC), Grafana, Superset
- **Production:** Replace training data with real DDC surveillance feeds

---

## Questions?

| Common Question | Answer |
|----------------|--------|
| Power BI? | Yes, via Vertica ODBC driver |
| Data scale? | Petabytes. DDC data is small for Vertica |
| Cloud? | AWS, GCP, Azure all supported |